In [2]:

secret_value_0 = "Bearer eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJ0cmFuc2FjdGlvbl9pZCI6IjM4NzhmNGViLTQ4ZDctNGQ1Yy04ZjNkLWFkZGNkMzkwNTcwMSIsInN1YiI6IjNlMGRhYjdiLWQxMmEtMTFmMC1hMDI3LWI3ZDIxZGNjZmJlNCIsImF1ZCI6WyJyZXN0c2VydmljZSJdLCJ1c2VyX25hbWUiOiJldmtpbmd3aW5kOTIxOEBnbWFpbC5jb20iLCJzY29wZSI6WyJyZWFkIl0sImlzcyI6Imh0dHBzOi8vbG9jYWxob3N0IiwibmFtZSI6ImV2a2luZ3dpbmQ5MjE4QGdtYWlsLmNvbSIsInV1aWRfYWNjb3VudCI6IjNlMGRhYjdiLWQxMmEtMTFmMC1hMDI3LWI3ZDIxZGNjZmJlNCIsImF1dGhvcml0aWVzIjpbIlVTRVIiLCJUUkFDS18yIl0sImp0aSI6IjcxNzc1YWMyLTNmMmMtNGRmOS1hMjhiLTU5ZTg2MjlhYjY0MSIsImNsaWVudF9pZCI6ImFkbWluYXBwIn0.qu79wk6jpTdGGu875nGoRb5e6bfNAhZHvyH7tp9oCSIXPDOvtrX2x3aITchcfgEbtAYE3fA-OjmRvZsP6juuTzPDWmpyUZerXxIPMEe5iKsVv-7GJMWxJSBfTqOrzFPPcflL1z-djrn8V3BHtItPnV1oKSd4UxKwK_C7qQbYt6QNSJfVlGXwEUxVfUGXN-3OKMfM-pSnUxIENrj2t8TQgm4zaEpVKAM9eoqej-eWrmi914GlfQF51A-kUi2d9Blxy85oFPbgrTYtb-aTU1LerthIuTzsoy9j0-X9TiWpTkT2Ch9EFzD7D-5InPR4ksCoFWO2juRh2fwgbQWM0clZng"
secret_value_1 = "4525a834-466f-6f29-e063-62199f0a0f81"
secret_value_2 = "MFwwDQYJKoZIhvcNAQEBBQADSwAwSAJBAJnt14kA1iKerSkBbpdzCG8Db9PR1RATzGrVc4ps3GbAgrs3hRHIJSWPhKOaPXRFhgpCn5ot1+1NGbJimOLpdf0CAwEAAQ=="

In [3]:
import requests
import json
import string
from tqdm import tqdm 
import time
# --- PHẦN 1: HÀM XỬ LÝ DATA & TẠO PROMPT (Đã tối ưu) ---
def create_user_content(item):
    # 1. Tách văn bản và câu hỏi
    raw_text = item['question']
    split_marker = "Câu hỏi:"
    parts = raw_text.split(split_marker)
    
    if len(parts) >= 2:
        van_ban = parts[0].replace("Đoạn thông tin:", "").strip()
        cau_hoi_text = parts[-1].strip()
    else:
        van_ban = raw_text
        cau_hoi_text = "Không tìm thấy câu hỏi tách biệt"

    # 2. Xử lý động Choices (A, B, C, D, E...)
    choices_list = item['choices']
    choices_dict = {}
    dynamic_evaluation_schema = {} 
    labels = string.ascii_uppercase 

    for idx, choice_text in enumerate(choices_list):
        if idx < len(labels):
            label = labels[idx]
            choices_dict[label] = choice_text
            dynamic_evaluation_schema[label] = {
                "ket_qua": "Đúng hoặc Sai",
                "ly_do": "Lý do cụ thể dựa trên văn bản"
            }

    # 3. Tạo cấu trúc JSON
    user_payload = {
        "van_ban": van_ban,
        "cau_hoi": cau_hoi_text,
        "lua_chon": choices_dict,
        "yeu_cau": {
            "nguyen_tac_cot_loi": [
            "1. KHÔNG BỊA ĐẶT: Mọi khẳng định phải có trích dẫn nguyên văn kèm index ĐÚNG của trích dẫn đó, không bịa index, index không chắc chắn nghĩa là trích dẫn không chắc chắn.",
            "2. KHÔNG SUY DIỄN: Chỉ dùng thông tin có trong văn bản (No outside knowledge).",
            "3. KHÔNG LƯỜI BIẾNG: Mọi trường 'index' phải điền số cụ thể từ văn bản, không suy diễn, không chú thích linh tinh, PHẢI TÌM ĐÚNG INDEX trong văn bản."
            ],
            "quy_trinh_tu_duy": {
                "buoc_0_phan_tich_cau_hoi": {
                    "mo_ta": "Bước định hướng tư duy: Xác định bẫy trong câu hỏi và tiêu chuẩn đúng/sai.",
                    "nhiem_vu": [
                        "0.0. Đọc kĩ câu hỏi một lần để biết được cần phải làm gì, ví dụ như chọn đúng nhất, chọn thấp nhất, chọn khác loại, ...",
                        "0.1. Xác định dạng câu hỏi: Tìm từ khóa dạng 'NGOẠI TRỪ', 'SAI' để hiểu rõ câu hỏi, thường các từ này sẽ in hoa. Nếu có -> Đánh dấu là Câu hỏi Phủ định.",
                        "0.2. Thiết lập tiêu chuẩn đánh giá:",
                        "   - 'Đúng': Khớp hoàn toàn ý nghĩa với văn bản.",
                        "   - 'Sai': Thông tin mâu thuẫn, hoặc chỉ đúng một nửa hoặc có nói tới nhưng không rõ ràng hẳn Ví dụ như nhắc tới một phần của câu nhưng không đầy đủ kiểu nói tới việc A nhưng không nói tới việc B trong A và hỏi là của việc B, hoặc văn bản nói khác.",
                        "   - 'Không có thông tin': Văn bản hoàn toàn không nhắc đến đối tượng/hành động này, nhấn mạnh là hoàn toàn không nhắc tới hành động đó."
                    ]
                },
                "buoc_1_truy_vet_va_lien_ket_ngu_canh": {
                    "mo_ta": "Đây là bước QUAN TRỌNG NHẤT. Phải giải mã mọi từ ngữ mơ hồ trước khi phân tích.",
                    "nhiem_vu": [
                        "1.1. Tìm và trích xuất đoạn văn bản chứa thông tin liên quan trực tiếp nhất.",
                        "1.2. Thực hiện 'GIẢI MÃ THAM CHIẾU' (Bắt buộc):",
                        "   - Quét đoạn trích, tìm tất cả các đại từ/từ chỉ thị (ví dụ: 'nó', 'hắn', 'điều này', 'khi đó', 'tại đây', 'việc ấy'...).",
                        "   - Truy ngược lại các câu trước đó để xác định chính xác danh từ gốc mà các từ này thay thế.",
                        "   - (Trong đầu): Tự viết lại câu văn với danh từ gốc đã thay thế vào vị trí đại từ.",
                        "1.3. Thực hiện bài kiểm tra 'Người lạ' (The Stranger Test):",
                        "   - Đưa đoạn trích (kèm chú thích giải mã ở bước 1.2) cho người chưa đọc văn bản gốc.",
                        "   - Nếu câu vẫn còn mơ hồ hoặc thiếu chủ ngữ thực sự -> BẮT BUỘC mở rộng trích dẫn thêm về phía trước.",
                        "1.4. Chốt đoạn trích cuối cùng: Phải là đoạn văn HOÀN CHỈNH, các đại từ đã rõ nghĩa."
                    ]
                },
                "buoc_2_phan_tich_va_loai_tru": {
                    "mo_ta": "So sánh logic chặt chẽ, chú ý các từ nối gây đảo ngược ý nghĩa. Index bắt buộc là số.",
                    "nhiem_vu": [
                        f"Đối chiếu từng lựa chọn {', '.join(choices_dict.keys())} với đoạn văn ĐÃ ĐƯỢC GIẢI MÃ ở Bước 1:",
                        "   - Kiểm tra Logic: Chú ý các từ nối (tuy nhiên, mặc dù, không phải, trừ khi...).",
                        "   - Đánh giá: Đúng, Sai hoặc Không có thông tin.",
                        "   - 'Đúng': Khớp hoàn toàn ý nghĩa với văn bản.",
                        "   - 'Sai': Thông tin mâu thuẫn, hoặc chỉ đúng một nửa, hoặc văn bản nói khác.",
                        "   - 'Không có thông tin': Văn bản hoàn toàn không nhắc đến đối tượng/hành động này.",
                        "   - Lý do: Phải trích dẫn được từ ngữ cụ thể trong văn bản. Nếu văn bản nói A, lựa chọn nói 'Không A' -> Sai.",
                        "   - XÁC ĐỊNH INDEX (BẮT BUỘC):",
                        "   - Phải tìm chính xác con số chỉ vị trí ký tự bắt đầu của Lý do trong Văn Bản gốc, không được bịa số.",
                        "   - TUYỆT ĐỐI KHÔNG ĐƯỢC ghi là 'không cần thiết' hay 'đã rõ'.",
                        "   - YÊU CẦU CỰC ĐOAN: Phải tìm ra một CON SỐ CỤ THỂ và chính xác. Không được nói 'khoảng', không được nói 'không tìm thấy' hay 'có thể không chính xác' nếu thông tin có đó. Phải đếm cho ra số thì mới được ghi dẫn chứng."
                    ]
                },
                "buoc_3_chon_dap_an": {
                    "mo_ta": "Xác định các đáp án phù hợp với câu hỏi.",
                    "nhiem_vu": "Dựa trên câu hỏi và các phân tích ở bước 2, chọn ra các đáp án phù hợp nhất với câu hỏi. Nếu như chỉ có một đáp án thì bỏ qua bước 4."
                },
                "buoc_4_ket_luan": {
                    "mo_ta": "Xác định đáp án duy nhất, NHẮC LẠI: DUY NHẤT MỘT ĐÁP ÁN ĐÚNG có dẫn chứng xuất hiện sớm nhất TRONG VĂN BẢN, không có chuyện B và C đều đúng, chỉ có DUY NHẤT MỘT đáp án đúng.",
                    "nhiem_vu": [
                        "0. Nhìn lại bước 3 để xem xem có bao nhiêu đáp án là đáp án của câu hỏi, nếu chỉ có một thì bỏ qua bước này.",
                        "1. Lấy trích dẫn nguyên văn (ở Bước 2) của các đáp án đó.",
                        "2. Rà soát văn bản gốc: Tìm vị trí bắt đầu của trích dẫn đó (ước lượng chỉ số index bắt đầu từ 0). Phải tìm cho bằng được chỉ số rõ ràng.",
                        "3. So sánh các chỉ số index: Số nhỏ hơn nghĩa là xuất hiện trước.",
                        "4. Kết luận: Chọn đáp án có chỉ số index NHỎ NHẤT."
                    ]
                }
            },
            "dinh_dang_tra_loi": {
                "buoc_0_phan_tich_cau_hoi": {
                    "mo_ta": "Xác định loại câu hỏi và tiêu chí",
                    "gia_tri": "Dạng câu hỏi: [Khẳng định/Phủ định] | Lưu ý: [Ghi chú đặc biệt]"
                },
                "buoc_1_doan_van_lien_quan": {
                    "mo_ta": "Trích nguyên văn đoạn văn bản (đã bao gồm câu trước/sau để đủ ngữ cảnh)",
                    "gia_tri": "string"
                },
                "buoc_1_ket_qua_stranger_test": {
                    "mo_ta": "Ghi rõ các từ thay thế đã được giải mã để người lạ hiểu. Cấu trúc: [Từ thay thế] = [Đối tượng gốc]",
                    "gia_tri": "Ví dụ: 'Nó' = 'Chiếc xe'; 'Lúc đó' = 'Năm 1990'; 'Ông ấy' = 'Tác giả'. -> Đã đủ ngữ cảnh."
                },
                "buoc_2_danh_gia_lua_chon": {
                    "mo_ta": "Đánh giá chi tiết từng lựa chọn",
                    "cau_truc_mau": {
                        "A": { "trang_thai": "...", "ly_do": "...", "index_dan_chung": "..." },
                        "B": "...",
                        "C": "...",
                        "D": "..."
                    },
                    "luu_y": ["XÁC ĐỊNH INDEX là BẮT BUỘC để chắc chắn dẫn chứng là đúng tuyệt đối:",
                        "   - Phải tìm chính xác con số chỉ vị trí ký tự bắt đầu của Lý do trong văn bản gốc.",
                        "   - TUYỆT ĐỐI KHÔNG ĐƯỢC ghi là 'không cần thiết' hay 'đã rõ'.",
                        "   - Nếu không đưa ra được con số cụ thể -> Coi như không tìm thấy dẫn chứng."
                             ]
                },
                "buoc_3_chon_dap_an": {
                    "mo_ta": "Liệt kê các đáp án đúng tìm thấy ở bước 2.",
                    "gia_tri": "Các đáp án phù hợp là: [...]. (Nếu chỉ có 1 đáp án thì ghi rõ)."
                },
                "buoc_4_ket_luan": {
                    "mo_ta": "So sánh chỉ số index (nếu có nhiều hơn 1 đáp án đúng) và chốt đáp án.",
                    "gia_tri": "Dựa trên Bước 2: Lý do của đáp án [...] là '...' bắt đầu từ kí tự thứ [...] trong văn bản gốc; Lý do của đáp án [...] là '...' bắt đầu từ kí tự thứ [...] trong văn bản gốc. -> Chọn [...]"
                },
                "dap_an_cuoi_cung": "/".join(choices_dict.keys())
            },
            "luu_y_quan_trong": [
                "KHÔNG được suy diễn hoặc dùng kiến thức thế giới",
                "TUYỆT ĐỐI KHÔNG BỊA SỐ, BỊA DỮ LIỆU "
                "Phân biệt rõ: 'Sai' là thông tin mâu thuẫn với văn bản hoặc văn bản nhắc tới nửa vời, còn 'Không có thông tin' là văn bản hoàn toàn không nhắc đến.",
                "Ưu tiên đáp án có dẫn chứng xuất hiện SỚM NHẤT trong văn bản khi có xung đột."
            ]
        }
    }
    return user_payload
# --- PHẦN 2: CHUẨN BỊ DỮ LIỆU ---
file_path = './data/test.json'

# Danh sách cần chạy
# TARGET_QIDS = [
#     'val_0001', 'val_0007', 'val_0009', 'val_0010', 'val_0011',
#     'val_0015', 'val_0019', 'val_0032', 'val_0045', 'val_0046',
#     'val_0048', 'val_0053', 'val_0060', 'val_0062', 'val_0063',
#     'val_0072', 'val_0074', 'val_0081', 'val_0088', 'val_0092'
# ]
results_buffer = [] # List chứa kết quả thô
try:
    with open(file_path,"r",encoding='utf-8') as f:
        data = json.load(f)
    data = [item for item in data if item['question'].startswith('Đoạn thông tin:')]
    # Chuyển data thành dict để tra cứu nhanh
    # data_map = {item['qid']: item for item in raw_data}

    # print(f"Bắt đầu xử lý {len(TARGET_QIDS)} câu hỏi...")

    # Dùng tqdm lặp qua danh sách
    # for qid in tqdm(TARGET_QIDS):
    #     # 1. Lấy item
    #     target_item = data_map.get(qid)
    #     if not target_item:
    #         print(f"Không tìm thấy qid: {qid}")
    #         continue
    for quest in data:
        # --------------------------------------------------------------------------------
        # 2. Tạo User Content & Đóng gói vào Instruction Wrapper
        # --------------------------------------------------------------------------------
        user_content_dict = create_user_content(quest)
        
        # CHÌA KHÓA: Nhúng toàn bộ dict cấu hình vào trong một prompt chỉ thị nghiêm ngặt
        user_content_str = f"""
        Bạn là một trợ lý AI tư duy logic nghiêm ngặt.
        Nhiệm vụ: Phân tích văn bản và câu hỏi dựa trên cấu hình JSON được cung cấp dưới đây.
        
        YÊU CẦU BẮT BUỘC VỀ ĐẦU RA (OUTPUT FORMAT):
        1. Chỉ trả về duy nhất một chuỗi JSON hợp lệ (Valid JSON String).
        2. KHÔNG thêm bất kỳ lời dẫn, giải thích hay markdown (như ```json) nào bên ngoài JSON.
        3. Cấu trúc JSON trả về phải khớp chính xác với schema được định nghĩa trong trường "dinh_dang_tra_loi" của cấu hình.
        4. Tuyệt đối tuân thủ các bước tư duy trong trường "yeu_cau".
        
        DƯỚI ĐÂY LÀ DỮ LIỆU ĐẦU VÀO VÀ CẤU HÌNH TƯ DUY:
        {json.dumps(user_content_dict, ensure_ascii=False, indent=2)}
        """

        # --------------------------------------------------------------------------------
        # 3. System Prompt (Cập nhật để nhấn mạnh vai trò JSON machine)
        # --------------------------------------------------------------------------------
        system_content_str = (
            "Bạn là một hệ thống xử lý ngôn ngữ tự nhiên chuyên biệt. "
            "Nhiệm vụ duy nhất của bạn là đọc dữ liệu đầu vào và trả về kết quả dưới dạng JSON chuẩn xác theo yêu cầu. "
            "Không được chat, không được giải thích, chỉ xuất JSON."
        )
        # 4. Cấu hình Header & Data
        headers = {
            'Authorization': f'{secret_value_0}',
            'Token-id': f'{secret_value_1}',
            'Token-key': f'{secret_value_2}',
            'Content-Type': 'application/json',
        }

        json_data = {
            'model': 'vnptai_hackathon_large', 
            'messages': [
                {'role': 'system', 'content': system_content_str},
                {'role': 'user', 'content': user_content_str},
            ],
            'temperature': 0.1,
            'top_p': 1.0,
            'top_k': 20,
            'n': 1,
            'max_completion_tokens': 10000,
        }

        # 5. Gọi API
        try:
            # Gửi đến endpoint large (đã sửa)
            response = requests.post('https://api.idg.vnpt.vn/data-service/v1/chat/completions/vnptai-hackathon-large', headers=headers, json=json_data)
            
            if response.status_code == 200:
                result = response.json()
                raw_response_content = result['choices'][0]['message']['content']
                print(f"ĐÃ HANDLE {quest['qid']} ✅")
                # Lưu kết quả thô vào buffer
                results_buffer.append({
                    "qid": quest['qid'],
                    "answer": raw_response_content
                })
            else:
                print(f"Lỗi API tại {quest['qid']}: {response.status_code} - {response.text}")
                
        except Exception as e:
            print(f"Lỗi request tại {quest['qid']}: {e}")
        
        # Nghỉ nhẹ 0.5s
        time.sleep(2)

    # 6. Lưu toàn bộ ra file
    output_file = 'raw_results.json'
    with open(output_file, 'w', encoding='utf-8') as f_out:
        json.dump(results_buffer, f_out, ensure_ascii=False, indent=2)

    print(f"\nĐã hoàn thành! Kết quả thô được lưu tại: {output_file}")
except Exception as e:
    print(f"Có lỗi hệ thống: {e}")

ĐÃ HANDLE test_0008 ✅
ĐÃ HANDLE test_0014 ✅
ĐÃ HANDLE test_0017 ✅
ĐÃ HANDLE test_0018 ✅
ĐÃ HANDLE test_0020 ✅
ĐÃ HANDLE test_0021 ✅
ĐÃ HANDLE test_0023 ✅
Lỗi request tại test_0024: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
ĐÃ HANDLE test_0028 ✅
ĐÃ HANDLE test_0042 ✅
ĐÃ HANDLE test_0043 ✅
ĐÃ HANDLE test_0044 ✅
ĐÃ HANDLE test_0046 ✅
ĐÃ HANDLE test_0059 ✅
ĐÃ HANDLE test_0069 ✅
ĐÃ HANDLE test_0070 ✅
ĐÃ HANDLE test_0071 ✅
ĐÃ HANDLE test_0077 ✅
ĐÃ HANDLE test_0083 ✅
ĐÃ HANDLE test_0091 ✅
ĐÃ HANDLE test_0096 ✅
ĐÃ HANDLE test_0098 ✅
ĐÃ HANDLE test_0105 ✅
ĐÃ HANDLE test_0109 ✅
ĐÃ HANDLE test_0111 ✅
ĐÃ HANDLE test_0118 ✅
ĐÃ HANDLE test_0120 ✅
ĐÃ HANDLE test_0128 ✅
ĐÃ HANDLE test_0133 ✅
ĐÃ HANDLE test_0139 ✅
ĐÃ HANDLE test_0141 ✅
ĐÃ HANDLE test_0142 ✅
ĐÃ HANDLE test_0143 ✅
ĐÃ HANDLE test_0146 ✅
ĐÃ HANDLE test_0150 ✅
ĐÃ HANDLE test_0152 ✅
ĐÃ HANDLE test_0154 ✅
ĐÃ HANDLE test_0164 ✅
ĐÃ HANDLE test_0166 ✅
ĐÃ HANDLE test_0175 ✅
ĐÃ HANDLE test_0178 ✅


In [ ]:
with open('raw_results.json','r',encoding='utf-8') as f:
    data = json.load(f)
with open('./data/test.json','r',encoding='utf-8') as f:
    sample = json.load(f)

In [5]:
import pandas as pd
with open('raw_results.json','r',encoding='utf-8') as f:
    data = json.load(f)
flat_data = []
for item in data:
    flat_item = {
        'qid': item['qid'],
        'answer': item['answer']
    }
    flat_data.append(flat_item)
df = pd.DataFrame(flat_data)
df.to_csv('results.csv', index=False, encoding='utf-8-sig')

In [4]:
import json

input_file = 'raw_results.json'

try:
    with open(input_file, 'r', encoding='utf-8') as f:
        results = json.load(f)

    print(f"Tổng số bản ghi tìm thấy: {len(results)}\n")

    for item in results:
        print(f"========== QID: {item['qid']} (Đáp án đúng: {item['ground_truth']}) ==========")
        print("--- NỘI DUNG AI TRẢ VỀ (RAW) ---")
        # In nguyên văn, không cắt gọt
        print(item['raw_answer']) 
        print("\n" + "="*80 + "\n")

except FileNotFoundError:
    print(f"Không tìm thấy file {input_file}. Kiểm tra lại xem đã chạy bước trước chưa.")
except Exception as e:
    print(f"Lỗi: {e}")

Không tìm thấy file raw_results.json. Kiểm tra lại xem đã chạy bước trước chưa.
